In [1]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [2]:
# 1. Load Raw Dataset
df = pd.read_csv('../dataset/initial/2_cve_vulnerabilities.csv') 


In [3]:

# 2. Feature Engineering: Calculate Patching Window (Days)
df['dateAdded'] = pd.to_datetime(df['dateAdded'])
df['dueDate'] = pd.to_datetime(df['dueDate'])
df['remediation_days'] = (df['dueDate'] - df['dateAdded']).dt.days

In [4]:
# 3. Categorical Grouping (Keep Top 10 + 'Other')
top_vendors = df['vendorProject'].value_counts().head(10).index
df['vendor_grouped'] = df['vendorProject'].apply(lambda x: x if x in top_vendors else 'Other')

top_cwes = df['cwes'].value_counts().head(10).index
df['cwe_grouped'] = df['cwes'].apply(lambda x: x if x in top_cwes else 'Other')



In [5]:
# 4. Map Target Variable (Known = 1, Unknown = 0)
y = df['knownRansomwareCampaignUse'].map({'Known': 1, 'Unknown': 0})

In [6]:
# 5. One-Hot Encoding
X_categorical = pd.get_dummies(df[['vendor_grouped', 'cwe_grouped']], drop_first=True, dtype=int)
X_numerical = df[['remediation_days']]

# Combine features into matrix X
X_raw = pd.concat([X_categorical, X_numerical], axis=1)



In [7]:
# 6. Stratified Train-Test Split (80/20)
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw, y, test_size=0.20, random_state=42, stratify=y
)



In [8]:
# 7. Apply Feature Scaling (StandardScaler)
scaler = StandardScaler()

X_train_scaled = X_train_raw.copy()
X_test_scaled = X_test_raw.copy()

X_train_scaled[['remediation_days']] = scaler.fit_transform(X_train_raw[['remediation_days']])
X_test_scaled[['remediation_days']] = scaler.transform(X_test_raw[['remediation_days']])

In [9]:
# 8. Export Processed Data to Disk
os.makedirs('../dataset/processed', exist_ok=True)

X_train_raw.to_csv('../dataset/processed/X_train_raw.csv', index=False)
X_test_raw.to_csv('../dataset/processed/X_test_raw.csv', index=False)
X_train_scaled.to_csv('../dataset/processed/X_train_scaled.csv', index=False)
X_test_scaled.to_csv('../dataset/processed/X_test_scaled.csv', index=False)
y_train.to_csv('../dataset/processed/y_train.csv', index=False)
y_test.to_csv('../dataset/processed/y_test.csv', index=False)

print("Preprocessing complete! Processed CSVs saved to '../dataset/processed/'.")

Preprocessing complete! Processed CSVs saved to '../dataset/processed/'.
